### Read experimental result data

In [7]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitter.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitter.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (524_288, 5)
┌─────┬───────┬───────┬───────┬───────────────┐
│ id  ┆ ip    ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---   ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64   ┆ u64   ┆ str   ┆ f64           │
╞═════╪═══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 9205  ┆ 0     ┆ 33000 ┆ 410.35        │
│ 0   ┆ 9205  ┆ 0     ┆ 23000 ┆ 500.61        │
│ 0   ┆ 9205  ┆ 0     ┆ 22000 ┆ 526.94        │
│ 0   ┆ 9205  ┆ 0     ┆ 32000 ┆ 536.62        │
│ 0   ┆ 9205  ┆ 0     ┆ 30000 ┆ 561.28        │
│ …   ┆ …     ┆ …     ┆ …     ┆ …             │
│ 511 ┆ 51500 ┆ 31    ┆ 33333 ┆ 16.38         │
│ 511 ┆ 51500 ┆ 31    ┆ 20323 ┆ 177.76        │
│ 511 ┆ 51500 ┆ 31    ┆ 11333 ┆ 37.89         │
│ 511 ┆ 51500 ┆ 31    ┆ 00133 ┆ 1009.68       │
│ 511 ┆ 51500 ┆ 31    ┆ 31033 ┆ 146.45        │
└─────┴───────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [8]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [66]:
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))#普通のdfにlf_optで事後確認した最適なmsgを他にip,us_idに適用されたのがあればそれをあわせて平均化。
    .group_by("ip_right", "us_id_right")#他のip,user_stateにもoptのmsgが通用するかという問い
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])#ipとus_idにピッタリなmsg
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias("ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値"),
             pl.col("max_" + MEAN_NUM_XACT).alias("ipとuser_stateに最適なmsg適用後の期待値")])
)
lf_opt_mod.mean().collect()

"ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値",ipとuser_stateに最適なmsg適用後の期待値
f64,f64
696.514736,1071.122441


ipだけぴったりのmsgだけ適用すると?

In [ ]:
ip_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["ip"]))
    .unique(["ip", MEAN_NUM_XACT])
    .select([pl.col("ip"), pl.col("msg"), pl.col(MEAN_NUM_XACT)])
).collect()


In [75]:
ip_opt

ip,msg,mean_num_xact
u64,str,f64
79977,"""00013""",5435.48
51976,"""12003""",50.14
775,"""00012""",8389.31
51500,"""00013""",18504.16
81019,"""00013""",815.04
…,…,…
68991,"""00113""",255.71
6257,"""20003""",3911.72
69029,"""00003""",340.71


In [77]:
ip_opt.select(pl.col("mean_num_xact").mean().alias("ip_opt_msg_xact"))#平均値がipとuser_stateより高い

ip_opt_msg_xact
f64
4233.55


In [117]:
ip_opt.group_by("msg").len()

msg,len
str,u32
"""10002""",1
"""00013""",4
"""00033""",1
"""12003""",1
"""00012""",1
"""20003""",2
"""00002""",1
"""00113""",1
"""00003""",4


In [110]:
ip_opt_list=(ip_opt.unique("msg").select(pl.col("msg")))["msg"].to_list()

In [111]:
ip_opt_list

['20003',
 '00013',
 '10002',
 '00002',
 '12003',
 '00113',
 '00012',
 '00003',
 '00033']

ipだけぴったりのmsgだけ適用すると外部行動者数が増加した

In [88]:
ip_opt_mod = (df.lazy()#ipにピッタリなmsgを他のシナリオに適用した場合。
    .join(ip_opt.lazy(), left_on=pl.col("msg"), right_on=pl.col("msg"))
    .group_by("ip_right").agg(pl.col("mean_num_xact").mean()).sort("mean_num_xact",descending=True)
    .select(pl.col("ip_right").alias("ip"),pl.col("mean_num_xact").alias("他のシナリオにmsgを適用したxact"))).collect()
ip_opt_frame=ip_opt_mod.join(ip_opt,left_on="ip",right_on="ip").select(pl.col("ip"),pl.col("msg"),pl.col("他のシナリオにmsgを適用したxact")).sort("他のシナリオにmsgを適用したxact",descending = True)
ip_opt_frame

ip,msg,他のシナリオにmsgを適用したxact
u64,str,f64
79222,"""00003""",1000.261758
28470,"""00003""",1000.261758
33655,"""00003""",1000.261758
69029,"""00003""",1000.261758
79977,"""00013""",962.443496
…,…,…
68991,"""00113""",746.384082
6257,"""20003""",739.66541
27644,"""20003""",739.66541


In [89]:
ip_opt_frame.select(pl.col("他のシナリオにmsgを適用したxact").mean().alias("他のシナリオにmsgを適用したxactの期待値"))

他のシナリオにmsgを適用したxactの期待値
f64
855.104987


In [ ]:
eval_df

key,value
str,f64
"""ev_by_opt""",696.514736
"""mean_of_opt_values""",1071.122441
"""ev_by_ram""",1000.261758
"""ev_by_urs""",226.414718
"""ram ev / opt ev""",1.436096
"""urs ev / opt ev""",0.325068
"""ram ev / opt mean""",0.933844
"""urs ev / opt mean""",0.211381


In [100]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details[0:16]

msg,mean_num_xact
str,f64
"""00003""",1000.261758
"""00013""",962.443496
"""00023""",894.841211
"""10003""",842.121113
"""00033""",836.626289
…,…
"""00113""",746.384082
"""20003""",739.66541
"""00032""",739.297188


In [14]:
lf_rob_details.select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
226.414718


In [15]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitter_user_analysis.parquet")
user_analysis


ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users,distance_from_ip
u64,u64,str,u64,f64,f64,u64
9205,0,"""33000""",1,0.0,0.21,2
9205,0,"""33000""",4,0.0,0.94,2
9205,0,"""33000""",5,0.0,0.04,3
9205,0,"""33000""",6,0.0,0.27,4
9205,0,"""33000""",7,0.0,0.64,2
…,…,…,…,…,…,…
51500,31,"""31033""",79863,0.1,0.0,3
51500,31,"""31033""",79955,0.01,0.0,4
51500,31,"""31033""",80280,0.31,0.0,3


In [126]:
ram_msg = (lf_rob_details.lazy().filter(pl.col("mean_num_xact")>800).select(pl.col("msg")).collect())
ram_msg_list =ram_msg["msg"].to_list()

ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

# # reverse_msg_list = lf_rob_details.lazy().sort("mean_num_xact",descending=True).select(pl.col("msg"))[-10:].to_list()
# # reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [130]:
ram_msg_list

['00002', '00003', '00012', '00013', '00023', '00033', '10003', '10013']

In [131]:
ip_opt_list

['20003',
 '00013',
 '10002',
 '00002',
 '12003',
 '00113',
 '00012',
 '00003',
 '00033']

00003,00033,00013,00002,00012は共通
つまり、少なくとも3ラウンドまで内部行動の一番強のメッセージを連続に伝播して、徐々に外部行動強のメッセージを伝播するのが外部行動者数を最大化してくれる。

In [18]:
reverse_msg_list

['33230',
 '23331',
 '32330',
 '23320',
 '33333',
 '33332',
 '33331',
 '33320',
 '23330',
 '33330']

In [19]:
ratio_xact_share_per = (user_analysis.lazy()
.select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")).collect())
xact_per_mean = (user_analysis.lazy().select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users")).collect())
share_per_mean = (user_analysis.lazy().select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users")).collect())

In [20]:
concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")
concat

mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.087129,0.137711,0.632694


In [21]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.140958,0.158284,0.890538


In [24]:
ram_xact_share= (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
).collect()
ram_xact_share

ip,us_id,msg,xact,share
u64,u64,str,f64,f64
79222,18,"""00003""",49.35,46.92
27644,19,"""00012""",26.92,55.23
28470,17,"""00013""",52.02,38.45
28470,14,"""00103""",70.32,50.66
68991,20,"""00002""",0.9,1.22
…,…,…,…,…
51976,2,"""00103""",4.32,3.72
79977,27,"""00003""",2952.92,4300.63
81019,1,"""00013""",269.56,227.65


In [27]:
ram_xact_share.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)

msg,xact,share
str,f64,f64
"""00003""",1000.261758,1210.570938
"""00013""",962.443496,940.822188
"""00023""",894.841211,706.027773
"""10003""",842.121113,963.967441
"""00033""",836.626289,538.170996
"""00002""",813.914824,1493.897773
"""10013""",807.108867,746.231914
"""00012""",802.23041,1177.989063
"""00022""",787.543867,932.753906


In [45]:
ram_ratio_mean = ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
1815.804544,0.262589,0.301526,0.900459


In [ ]:
re_ram_msg = pl.read_parquet("../../experiments/twitter_81306/reverse_msg_us_an.parquet")
action_stats_per_msg = (re_ram_msg.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).group_by("msg").agg(
    pl.col("num_xact_of_users").mean(),
    pl.col("num_share_of_users").mean())
)
ram =lf_rob_details.join(action_stats_per_msg.collect(),left_on="msg", right_on="msg").sort(pl.col("mean_num_xact"), descending=True)
re_ram_ratio = ram.lazy().select(pl.col("msg"),pl.col("mean_num_xact"),pl.col("num_xact_of_users"),pl.col("num_share_of_users"),(pl.col("num_xact_of_users") / pl.col("num_share_of_users")).alias("xact_share_ratio")).collect()
re_ram_ratio

msg,mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64
"""13333""",176.582812,0.200084,0.208078,0.96158
"""02233""",175.57825,0.114405,0.280044,0.408525
"""01332""",171.856437,0.101695,0.294872,0.344879
"""01233""",169.0435,0.08959,0.292721,0.306059
"""00332""",168.221312,0.078627,0.307673,0.255555
"""00233""",163.912813,0.069287,0.303262,0.228472
"""02333""",144.857687,0.12186,0.269813,0.451648
"""03333""",142.906688,0.151186,0.257636,0.586821
"""01333""",139.436688,0.094801,0.281407,0.336882


['30000',
 '31000',
 '32000',
 '30001',
 '20000',
 '22000',
 '21000',
 '33000',
 '31001',
 '30100']

In [46]:
re_ram_ratio_mean = re_ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
re_ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
159.080519,0.109499,0.279096,0.412902


逆の順番に情報を提供すると

In [16]:
re_ram_xact_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_xact_of_users").mean())
re_ram_share_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_share_of_users").mean())

re_ram_xact_mean.collect()
re_ram_share_mean.collect()

num_share_of_users
f64
0.315331


最適戦略

拡散モデルip,us_idの組み合わせに、それに向ける最適解のmsgを同時に適用した全てのリストのユーザー状態

In [5]:
opt_df = (
    df.lazy()
    .filter(pl.col("mean_num_xact") == pl.col("mean_num_xact").max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", "mean_num_xact"])
    .sort(pl.col("mean_num_xact"),descending = True)
    .select(pl.col("us_id"),pl.col("ip"),pl.col("msg"))
)
# 2. 結合の「キー」として使う LazyFrame を準備
#    opt_df の "argmax_msg" を user_analysis の "msg" に名前を合わせる
opt_keys = opt_df.lazy().select(
    pl.col("us_id"),
    pl.col("ip"),
    pl.col("msg")
)
# 3. user_analysis を 'semi' join でフィルタリングする
opt_ratio_lazy = user_analysis.lazy().join(
    opt_keys,
    on=["us_id", "ip", "msg"],  # 3つのキーがすべて一致する行を探す
    how="semi"                  # user_analysis 側に存在する行だけを残す
).collect()
opt_ratio_lazy


: 

In [7]:
opt_ratio_lazy = pl.read_parquet("../../experiments/twitter_81306/opt/opt_ratio_lazy.parquet")
opt_ratio_lazy

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
9199,0,"""30011""",1,0.0,0.02
9199,0,"""30011""",4,0.0,0.14
9199,0,"""30011""",6,0.0,0.03
9199,0,"""30011""",7,0.03,0.07
9199,0,"""30011""",9,0.04,0.0
…,…,…,…,…,…
68985,15,"""30000""",79824,0.0,0.01
68985,15,"""30000""",80119,0.07,0.07
68985,15,"""30000""",80265,0.39,0.5


各拡散モデルにおけるOptimalのmsgの結果のユーザー状態

ユーザー1人の平均外部・内部行動回数と外部対内部の比率

In [5]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""30010""",13,79973,0.134906,0.148113,0.910828
"""30000""",5,69023,0.06342,0.051303,1.23619
"""30000""",4,79973,0.047638,0.054216,0.878664
"""30002""",13,12605,0.02523,0.023032,1.095413
"""32221""",3,33649,0.45,0.71,0.633803
…,…,…,…,…,…
"""21113""",8,68985,0.226,0.247333,0.913747
"""33000""",11,38598,0.388505,0.373066,1.041383
"""30103""",9,9199,0.057205,0.092773,0.61661
